# Etap 3 – Baza SQL

Tworzenie bazy SQLite, ładowanie danych z `data/processed/clinical_processed.csv` i eksploracja przez zapytania SQL.

**Źródło danych:** `clinical_processed.csv` (1047 pacjentów, 19 kolumn)
**Baza:** `db/tcga_glioma.db`

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Ścieżki
DB_PATH = Path('../db/tcga_glioma.db')          # gdzie powstanie plik bazy
CSV_PATH = Path('../data/processed/clinical_processed.csv')
SCHEMA_PATH = Path('../sql/schema.sql')

DB_PATH.parent.mkdir(exist_ok=True)

print("Ścieżki OK")
print(f"  Baza:   {DB_PATH}")
print(f"  Dane:   {CSV_PATH}")
print(f"  Schema: {SCHEMA_PATH}")

Ścieżki OK
  Baza:   ..\db\tcga_glioma.db
  Dane:   ..\data\processed\clinical_processed.csv
  Schema: ..\sql\schema.sql


## 1. Tworzenie bazy i ładowanie schematu

In [2]:
with open(SCHEMA_PATH, 'r') as f:
    schema_sql = f.read()

conn = sqlite3.connect(DB_PATH)

conn.executescript(schema_sql)
conn.commit()

print("Baza utworzona, tabele gotowe")

OperationalError: table patients already exists

In [4]:
# Wyczyść tabele przed ponownym ładowaniem
# (na wypadek gdyby komórka była uruchomiona więcej niż raz)
conn.executescript("""
    DELETE FROM biomarkers;
    DELETE FROM survival;
    DELETE FROM patients;
""")
conn.commit()
print("Tabele wyczyszczone")

Tabele wyczyszczone


## 2. Ładowanie danych do tabel

In [5]:

df = pd.read_csv(CSV_PATH)


df_patients = df[['patient_id', 'study', 'histology', 'grade',
                   'age_at_diagnosis', 'gender', 'kps', 'mutation_count']]

df_biomarkers = df[['patient_id', 'idh_status', 'mgmt_status', 'codel_1p19q',
                     'idh_codel_subtype', 'tert_promoter_status', 'atrx_status',
                     'idh_mutant', 'mgmt_methylated', 'codel']]

df_survival = df[['patient_id', 'os_months', 'os_event']]


df_patients.to_sql('patients', conn, if_exists='append', index=False)
df_biomarkers.to_sql('biomarkers', conn, if_exists='append', index=False)
df_survival.to_sql('survival', conn, if_exists='append', index=False)

conn.commit()

print(f"Załadowano pacjentów:  {len(df_patients)}")
print(f"Załadowano biomarkerów: {len(df_biomarkers)}")
print(f"Załadowano survival:    {len(df_survival)}")

Załadowano pacjentów:  1047
Załadowano biomarkerów: 1047
Załadowano survival:    1047


## 3. Weryfikacja załadowanych danych

In [7]:


# Sprawdź liczbę wierszy w każdej tabeli
for tabela in ['patients', 'biomarkers', 'survival']:
    wynik = pd.read_sql(f"SELECT COUNT(*) AS liczba_wierszy FROM {tabela}", conn)
    print(f"{tabela}: {wynik['liczba_wierszy'][0]} wierszy")

patients: 1047 wierszy
biomarkers: 1047 wierszy
survival: 1047 wierszy
